# Inspect export parameters from Curry 9 .cdt files of an EEG database

This notebook inspects Curry 9 recording parameters across an entire dataset **without loading signal data**.
It reads the `.cdt.dpo` parameter file (a plain-text sidecar, ~7 kB) for each recording.

**Checks performed:**
- Channel configuration and montage consistency across recordings
- Sampling frequency consistency
- Signal unit consistency
- **Impedance check**: flags channels above the impedance threshold at recording start

**Not applicable for Curry float data** (unlike EDF): inverted polarity, clipping via dynamic range, digital resolution.

> Run with: `voila tools_curry/1_inspect_curry_voila.ipynb`

<hr style="height:4px; background-color:black; border:none;">
First, check that all required packages are installed.

In [ ]:
#%% Imports and shared functions
try:
    import os
    import sys
    import warnings
    import traceback
    import pandas as pd
    from pathlib import Path
    import ipywidgets as widgets
    from ipyfilechooser import FileChooser
    from IPython.display import display, HTML
    # curry shared modules — located next to this notebook
    _here = os.path.dirname(os.path.abspath('__file__'))
    if _here not in sys.path:
        sys.path.insert(0, _here)
    from curry_header import read_curry_header, get_impedance_summary
except ImportError as e:
    print('\u26a0\ufe0f Error: ', e)
else:
    print('\u2705 Packages and functions successfully imported!')


def _merge_save(new_df, path, key_col):
    """Merge new_df into an existing TSV on key_col (replace matching rows, keep others)."""
    if os.path.exists(path) and not new_df.empty:
        try:
            old = pd.read_csv(path, sep='\t', dtype=str)
            old = old.loc[:, ~old.columns.str.startswith('Unnamed')]
            if key_col in old.columns and key_col in new_df.columns:
                new_keys = set(os.path.normcase(str(k)) for k in new_df[key_col])
                old = old[~old[key_col].apply(lambda k: os.path.normcase(str(k)) in new_keys)]
                merged = pd.concat([old, new_df], ignore_index=True)
            else:
                merged = new_df
        except Exception:
            merged = new_df
    else:
        merged = new_df
    merged.to_csv(path, sep='\t', index=False)


def print_in_scrollable_box(text, height=300, font_size='12px'):
    display(HTML(
        f'<pre style="overflow-y:scroll; height:{height}px; border:1px solid black; '
        f'padding:10px; font-size:{font_size};">{text}</pre>'
    ))


#%% --- Section 1 UI ---
section1 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>1. Select your data folder</h2>
""")
display(section1)

chooser = FileChooser(os.getcwd())
chooser.title = '<b>Choose your study folder (containing .cdt files)</b>'
chooser.show_only_dirs = True

impedance_threshold_w = widgets.BoundedFloatText(
    value=20.0, min=0.1, max=500.0, step=1.0,
    description='Impedance flag threshold (k\u03a9):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px'),
)

skip_existing = widgets.Checkbox(
    value=True,
    description='Skip files already inspected',
    style={'description_width': 'initial'},
)
existing_files_info = widgets.HTML(value='')
run_button = widgets.Button(description='Run inspection', button_style='success', icon='play')
out = widgets.Output()


def _update_existing_info(*_):
    if not getattr(chooser, 'selected_path', None):
        existing_files_info.value = ''
        return
    try:
        folder = Path(chooser.selected_path)
        cdt_files = list(folder.rglob('*.cdt'))
        # exclude .cdt.dpo / .cdt.ceo etc — keep only bare .cdt
        cdt_files = [f for f in cdt_files if f.suffix == '.cdt' and not f.name.startswith('._')]
        n_total = len(cdt_files)
        if n_total == 0:
            existing_files_info.value = '<small style="color:#888;">No .cdt files found in selected folder.</small>'
            return
        full_path = folder / 'summary_inspection_curry' / 'FULL_summary_table_curry.tsv'
        n_done = 0
        if full_path.exists():
            df_prev = pd.read_csv(str(full_path), sep='\t', dtype=str)
            df_prev = df_prev.loc[:, ~df_prev.columns.str.startswith('Unnamed')]
            if 'path' in df_prev.columns:
                existing_paths = set(os.path.normcase(p) for p in df_prev['path'].astype(str))
                n_done = sum(os.path.normcase(str(f)) in existing_paths for f in cdt_files)
        n_new = (n_total - n_done) if skip_existing.value else n_total
        existing_files_info.value = (
            f'<small style="color:#555;">{n_done} / {n_total} .cdt file(s) already inspected. '
            f'{n_new} file(s) will be read on run.</small>'
        )
    except Exception as e:
        existing_files_info.value = f'<small style="color:#c0392b;">Error: {e}</small>'


chooser.register_callback(_update_existing_info)
skip_existing.observe(_update_existing_info, names='value')

display(chooser)
display(widgets.HBox([impedance_threshold_w]))
display(widgets.HBox([skip_existing, existing_files_info]))
display(run_button)
display(out)


# -----------------------------------------------------------------------
def run_inspection(_):
    out.clear_output()
    with out:
        try:
            _run_inspection_inner()
        except Exception:
            print('\u274c Unexpected error during inspection:')
            traceback.print_exc()


def _run_inspection_inner():
    if not getattr(chooser, 'selected_path', None):
        print('\u26a0\ufe0f Please select a folder first.')
        return

    folder_path = chooser.selected_path
    imp_threshold = impedance_threshold_w.value
    print(f'\U0001f4c1 Selected Path: {folder_path}')

    # Discover .cdt files (bare .cdt only — exclude .cdt.dpo / .cdt.ceo)
    cdt_files = [
        f for f in Path(folder_path).rglob('*.cdt')
        if f.suffix == '.cdt' and not f.name.startswith('._')
    ]
    if not cdt_files:
        print('\u26a0\ufe0f No .cdt files found in the selected folder.')
        return
    print(f'Found {len(cdt_files)} .cdt file(s).')

    # Summary output folder
    summary_path = os.path.join(folder_path, 'summary_inspection_curry')
    os.makedirs(summary_path, exist_ok=True)

    # Skip/merge setup
    full_table_path = os.path.join(summary_path, 'FULL_summary_table_curry.tsv')
    existing_full = pd.DataFrame()
    if os.path.exists(full_table_path):
        try:
            existing_full = pd.read_csv(full_table_path, sep='\t', dtype=str)
            existing_full = existing_full.loc[:, ~existing_full.columns.str.startswith('Unnamed')]
        except Exception as _e:
            print(f'\u26a0 Could not read existing table ({_e}); rebuilding from scratch.')
    existing_paths = (
        set(os.path.normcase(p) for p in existing_full['path'].astype(str))
        if 'path' in existing_full.columns else set()
    )

    to_read = [
        f for f in cdt_files
        if not skip_existing.value or os.path.normcase(str(f)) not in existing_paths
    ]
    n_skipped = len(cdt_files) - len(to_read)
    print(f'Skip already inspected: {"ON" if skip_existing.value else "OFF"}')
    print(f'Files to read this run: {len(to_read)} / {len(cdt_files)}'
          + (f' ({n_skipped} skipped)' if n_skipped else ''))
    if not to_read:
        print('\nNothing new to inspect. Displaying existing results below.')

    # -----------------------------------------------------------------------
    # MAIN LOOP
    # -----------------------------------------------------------------------
    df_list = []
    failed_list = []
    imp_flag_rows = []   # per-channel impedance flags
    output_log = ''
    dynamic_out = widgets.Output()
    display(dynamic_out)

    for e, cdt_path in enumerate(to_read):
        try:
            with dynamic_out:
                output_log += f'file {e+1}/{len(to_read)}: {cdt_path}\n'
                dynamic_out.clear_output(wait=True)
                print_in_scrollable_box(output_log, font_size='12px')

            hdr = read_curry_header(str(cdt_path))
            sub_name = hdr['file_id']
            sub_folder = cdt_path.parent.name

            # --- Per-channel rows for the full summary table ---
            for i, label in enumerate(hdr['all_ch_labels']):
                group_type = 'EEG' if i < hdr['eeg_group_size'] else 'other'
                df_list.append({
                    'subject':       sub_name,
                    'sub_folder':    sub_folder,
                    'path':          str(cdt_path),
                    'start_datetime': hdr['start_datetime'],
                    'sfreq':         hdr['sfreq'],
                    'n_samples':     hdr['n_samples'],
                    'duration_sec':  round(hdr['duration_sec'], 1),
                    'n_epochs_30s':  hdr['n_epochs_30s'],
                    'data_unit':     hdr['data_unit'],
                    'file_version':  hdr['file_version'],
                    'channel':       label,
                    'channel_group': group_type,
                    'n_eeg_channels': hdr['eeg_group_size'],
                    'n_other_channels': hdr['other_group_size'],
                })

            # --- Impedance flags ---
            for imp in get_impedance_summary(hdr):
                last = imp['last_kOhm']
                flagged = (last is not None) and (last > imp_threshold)
                imp_flag_rows.append({
                    'subject':      sub_name,
                    'path':         str(cdt_path),
                    'channel':      imp['label'],
                    'group':        imp['group'],
                    'last_kOhm':    last,
                    'flagged':      flagged,
                    'threshold_kOhm': imp_threshold,
                })

        except Exception as exc:
            failed_list.append({'path': str(cdt_path), 'error': str(exc)})
            with dynamic_out:
                output_log += f'  \u274c FAILED: {exc}\n'
                dynamic_out.clear_output(wait=True)
                print_in_scrollable_box(output_log, font_size='12px')

    # -----------------------------------------------------------------------
    # SAVE OUTPUTS
    # -----------------------------------------------------------------------
    print('\n\U0001f4be Saving outputs...')
    new_df = pd.DataFrame(df_list) if df_list else pd.DataFrame()

    # FULL summary table (cumulative, merged)
    _merge_save(new_df, full_table_path, key_col='path')
    print(f'  FULL_summary_table_curry.tsv updated ({full_table_path})')

    # Impedance flags
    if imp_flag_rows:
        imp_df = pd.DataFrame(imp_flag_rows)
        imp_path = os.path.join(summary_path, 'impedance_flags_curry.tsv')
        _merge_save(imp_df, imp_path, key_col='path')
        n_flagged = imp_df['flagged'].sum()
        print(f'  impedance_flags_curry.tsv: {n_flagged} channel(s) above {imp_threshold} k\u03a9')

    # Failed files
    if failed_list:
        fail_df = pd.DataFrame(failed_list)
        fail_path = os.path.join(summary_path, 'failed_cdt_read.tsv')
        _merge_save(fail_df, fail_path, key_col='path')
        print(f'  \u26a0 {len(failed_list)} file(s) failed — see failed_cdt_read.tsv')

    # -----------------------------------------------------------------------
    # DISPLAY RESULTS (this run only)
    # -----------------------------------------------------------------------
    # Reload full cumulative table for display
    try:
        full_df = pd.read_csv(full_table_path, sep='\t', dtype=str)
        full_df = full_df.loc[:, ~full_df.columns.str.startswith('Unnamed')]
    except Exception:
        full_df = new_df

    display(widgets.HTML('<hr style="height:2px; background-color:#aaa;"><h3>2. Inspection results</h3>'))

    if not full_df.empty:
        eeg_df = full_df[full_df['channel_group'] == 'EEG'].copy()
        display(widgets.HTML('<h4>2.1 Sampling frequency</h4>'))
        if 'sfreq' in full_df.columns:
            sf_configs = (
                full_df[['subject', 'sfreq']]
                .drop_duplicates()
                .groupby('sfreq')['subject']
                .apply(list)
                .reset_index()
            )
            sf_configs.columns = ['sfreq_Hz', 'subjects']
            sf_configs['n_subjects'] = sf_configs['subjects'].apply(len)
            display(sf_configs[['sfreq_Hz', 'n_subjects', 'subjects']])

        display(widgets.HTML('<h4>2.2 EEG channel configurations</h4>'))
        if not eeg_df.empty and 'channel' in eeg_df.columns:
            configs = (
                eeg_df.groupby('subject')['channel']
                .apply(lambda x: tuple(sorted(x)))
                .reset_index()
                .rename(columns={'channel': 'ch_set'})
            )
            config_groups = (
                configs.groupby('ch_set')['subject']
                .apply(list)
                .reset_index()
            )
            config_groups.columns = ['channel_set', 'subjects']
            config_groups['n_channels'] = config_groups['channel_set'].apply(len)
            config_groups['n_subjects'] = config_groups['subjects'].apply(len)
            display(config_groups[['n_channels', 'n_subjects', 'channel_set', 'subjects']])

        display(widgets.HTML('<h4>2.3 Data unit</h4>'))
        if 'data_unit' in full_df.columns:
            unit_configs = (
                full_df[['subject', 'data_unit']]
                .drop_duplicates()
                .groupby('data_unit')['subject']
                .apply(list)
                .reset_index()
            )
            unit_configs.columns = ['data_unit', 'subjects']
            unit_configs['n_subjects'] = unit_configs['subjects'].apply(len)
            display(unit_configs[['data_unit', 'n_subjects', 'subjects']])

        display(widgets.HTML(f'<h4>2.4 Impedance check (threshold: {imp_threshold} k\u03a9)</h4>'))
        imp_path = os.path.join(summary_path, 'impedance_flags_curry.tsv')
        if os.path.exists(imp_path):
            imp_full = pd.read_csv(imp_path, sep='\t', dtype={'flagged': bool})
            flagged_only = imp_full[imp_full['flagged'] == True]
            if flagged_only.empty:
                display(widgets.HTML(
                    f'<p style="color:green;">\u2705 All channels below {imp_threshold} k\u03a9 '
                    f'across all inspected recordings.</p>'
                ))
            else:
                display(widgets.HTML(
                    f'<p style="color:#c0392b;">\u26a0 {len(flagged_only)} channel/recording '
                    f'combination(s) above {imp_threshold} k\u03a9:</p>'
                ))
                display(flagged_only[['subject', 'channel', 'group', 'last_kOhm', 'threshold_kOhm']])

    print(f'\n\u2705 Done. Outputs written to: {summary_path}')


run_button.on_click(run_inspection)
